# Exploratory Data Analysis & Feature Engineering Comparison

In this notebook, we compare the correlation patterns in the Maternal Health Risk dataset before and after applying clinical feature engineering.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# Set Seaborn theme for premium aesthetics
sns.set_theme(style="whitegrid", palette="magma")

# Load the real dataset
data_path = os.path.join("..", "data", "Maternal Health Risk Data Set.csv")
df = pd.read_csv(data_path)

# Encode RiskLevel for correlation analysis
risk_map = {'low risk': 0, 'mid risk': 1, 'high risk': 2}
df_encoded = df.copy()
df_encoded['RiskLevel'] = df_encoded['RiskLevel'].map(risk_map)

print("Data loaded and target encoded.")
df.head()

## 1. Class Distribution

Using Seaborn's `countplot` to visualize the frequency of each risk category.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='RiskLevel', order=['low risk', 'mid risk', 'high risk'])
plt.title("Distribution of Maternal Health Risk Levels", fontsize=15)
plt.ylabel("Count (Patients)")
plt.show()

## 2. Correlation Matrix: BEFORE Feature Engineering

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df_encoded.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title("Correlation Matrix (Baseline Features)", fontsize=15)
plt.show()

## 3. Applying Feature Engineering

We apply clinical metrics such as Mean Arterial Pressure (MAP), Pulse Pressure, and Shock Index.

In [ ]:
# Clinical Feature Engineering
df_encoded["MAP"] = (df_encoded["SystolicBP"] + 2 * df_encoded["DiastolicBP"]) / 3
df_encoded["PulsePressure"] = df_encoded["SystolicBP"] - df_encoded["DiastolicBP"]
df_encoded["ShockIndex"] = df_encoded["HeartRate"] / df_encoded["SystolicBP"]
df_encoded["BPRatio"] = df_encoded["SystolicBP"] / df_encoded["DiastolicBP"]

df_encoded["TempDeviation"] = df_encoded["BodyTemp"] - 98.2
df_encoded["BSDeviation"] = df_encoded["BS"] - 7

df_encoded["CombinedRiskScore"] = (
    (df_encoded["MAP"] > 105).astype(int) +
    (df_encoded["BS"] > 10).astype(int) +
    (df_encoded["HeartRate"] > 90).astype(int) +
    (df_encoded["TempDeviation"] > 1).astype(int)
)

print("Feature engineering complete.")

## 4. Correlation Matrix: AFTER Feature Engineering

In [ ]:
plt.figure(figsize=(15, 10))
sns.heatmap(df_encoded.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title("Correlation Matrix (Engineered Features)", fontsize=15)
plt.show()